# 01. Inventário do banco medalhão do Lab: onde estão os dados da curva S

Banco `10.0.0.50/cidades`: Postgres medalhão do Gov Hub (schemas `bronze`, `prata`, `ouro`, `seeds`, `lake`), materializado pelo dbt do `data-application-cidades`. É aqui que vivem a série mensal por APF do FAR e a curva S agregada (issue #163).

Pergunta deste notebook: que tabelas usar para (1) a régua da curva S, (2) o rótulo oficial da CAIXA e (3) o histórico de paralisações.

In [1]:
import sys; sys.path.insert(0, "../..")
import pandas as pd
from src import db
pd.set_option("display.width", 200); pd.set_option("display.max_rows", 300); pd.set_option("display.max_colwidth", 80)
print(db.HOST)
db.schemas()

10.0.0.50


,schema,objetos
0,abecip,1
1,bronze,166
2,conjuntura,0
3,duckdb,2
4,fgv,1
5,ibge,14
6,lake,8
7,metadata,1
8,ouro,172
9,prata,141


## 1. Famílias de tabelas por schema

In [2]:
obj = db.ler_sql("""
select n.nspname as schema, c.relname as tabela, c.reltuples::bigint as linhas_est
from pg_class c join pg_namespace n on n.oid = c.relnamespace
where n.nspname in ('bronze','prata','ouro','seeds','lake') and c.relkind in ('r','v','m')
order by 1, 2""")
obj["familia"] = obj.tabela.str.extract(r"^(?:prata_|ouro_|bronze_)?([a-z]+)")[0]
obj.groupby(["schema", "familia"]).size().unstack(0).fillna(0).astype(int)

schema,bronze,ouro,prata,seeds
familia,,,,
abecip,3,0,0,0
api,0,0,0,1
bacen,2,0,0,0
boletim,0,0,0,1
campos,0,0,0,1
cbic,1,0,0,0
cobertura,0,0,0,2
colunas,0,0,0,2
conjuntura,0,54,36,0


## 2. A série do FAR já pronta (ouro)

`ouro_far_serie_mensal_apf`: uma linha por APF e mês, com físico realizado e previsto (MONIT, CAIXA ou carregado), situação MONIT e CAIXA, liberações do livro-razão, percentual financeiro e o ritmo físico x financeiro. `ouro_far_curva_s`: agregada por mês e nível (nacional, UF, portaria, município), ponderada pelo valor FAR.

In [3]:
for t, mes, key in [("ouro.ouro_far_serie_mensal_apf", "mes", "apf"), ("ouro.ouro_far_curva_s", "mes", None), ("prata.prata_far_caixa_serie", "competencia", "apf")]:
    extra = f", count(distinct {key}) n_apf" if key else ""
    print(t)
    display(db.ler_sql(f"select count(*) n, min({mes}) ini, max({mes}) fim, count(distinct {mes}) n_meses{extra} from {t}"))

ouro.ouro_far_serie_mensal_apf


,n,ini,fim,n_meses,n_apf
0,19070,2024-02-01,2026-08-01,31,886


ouro.ouro_far_curva_s


,n,ini,fim,n_meses
0,10100,2025-04-01,2026-08-01,17


prata.prata_far_caixa_serie


,n,ini,fim,n_meses,n_apf
0,111112,2024-06-01,2026-07-01,26,4505


In [4]:
db.colunas("ouro", "ouro_far_serie_mensal_apf")

,column_name,data_type
0,apf,text
1,nome_empreendimento,text
2,apf_municipio_empreendimento,text
3,municipio,text
4,uf,text
5,ic_cadastro_monit,boolean
6,mes,date
7,mes_label,text
8,mes_label_br,text
9,dt_inicio_obra,date


In [5]:
serie = db.ler_sql("""
select apf, uf, mes, dt_inicio_obra, etapa, pct_obra_realizada, pct_obra_prevista, fonte_fisica, situacao_obra,
       vr_liberado_acum, vr_emprestimo_far, pct_financeiro, ritmo_fisico_financeiro, situacao_caixa, pct_execucao_caixa
from ouro.ouro_far_serie_mensal_apf
where apf in (select apf from ouro.ouro_far_serie_mensal_apf group by apf order by random() limit 1)
order by mes""")
serie

,apf,uf,mes,dt_inicio_obra,etapa,pct_obra_realizada,pct_obra_prevista,fonte_fisica,situacao_obra,vr_liberado_acum,vr_emprestimo_far,pct_financeiro,ritmo_fisico_financeiro,situacao_caixa,pct_execucao_caixa
0,63175045,RN,2024-08-01,None,Aguardando início físico,NaN,NaN,None,None,0.0,36000000.0,0.00,None,None,NaN
1,63175045,RN,2024-09-01,None,Aguardando início físico,NaN,NaN,None,None,0.0,36000000.0,0.00,None,NORMAL,0.0
2,63175045,RN,2024-10-01,None,Aguardando início físico,NaN,NaN,None,None,0.0,36000000.0,0.00,None,EM ANDAMENTO,0.0
3,63175045,RN,2024-11-01,None,Aguardando início físico,NaN,NaN,None,None,0.0,36000000.0,0.00,None,EM ANDAMENTO,0.0
4,63175045,RN,2024-12-01,None,Aguardando início físico,NaN,NaN,None,None,0.0,36000000.0,0.00,None,EM ANDAMENTO,0.0
5,63175045,RN,2025-01-01,None,Aguardando início físico,NaN,NaN,None,None,0.0,36000000.0,0.00,None,EM ANDAMENTO,0.0
6,63175045,RN,2025-02-01,None,Aguardando início físico,NaN,NaN,None,None,0.0,36000000.0,0.00,None,EM ANDAMENTO,0.0
7,63175045,RN,2025-03-01,None,Aguardando início físico,NaN,NaN,None,None,0.0,36000000.0,0.00,None,EM ANDAMENTO,0.0
8,63175045,RN,2025-04-01,None,Aguardando início físico,0.0,NaN,caixa,None,0.0,36000000.0,0.00,Ritmo Equilibrado,EM ANDAMENTO,0.0
9,63175045,RN,2025-05-01,None,Aguardando início físico,0.0,NaN,caixa,None,0.0,36000000.0,0.00,Ritmo Equilibrado,EM ANDAMENTO,0.0


In [6]:
db.ler_sql("select situacao_obra, situacao_caixa, count(*) n from ouro.ouro_far_serie_mensal_apf where mes = (select max(mes) from ouro.ouro_far_serie_mensal_apf) group by 1,2 order by 3 desc")

,situacao_obra,situacao_caixa,n
0,NORMAL,None,439
1,EM ANDAMENTO,None,171
2,ATRASADA,None,119
3,NÃO INICIADA,None,104
4,CONCLUÍDA,None,36
5,None,None,8
6,OUTRAS,None,5
7,PARALISADA,None,4


## 3. O rótulo oficial da CAIXA (GEHIS, semanal desde 2021)

`bronze.bronze_sftp_gehis_andamento_obra` empilha os arquivos `CAIXA_AF_GEHIS_ANDAMENTO_OBRA_M*` (texto puro, com colunas quebradas de arquivos mal formados). Por APF e anomes traz `situacao_obra` no vocabulário da própria CAIXA. Na prata só existe a versão do Rural (`prata_rural_gehis_andamento_obra`); para o FAR vamos tipar a partir da bronze.

In [7]:
db.ler_sql("""
select upper(trim(situacao_obra)) as situacao_obra, count(*) n, count(distinct nu_apf) n_apf,
       min(coalesce(nullif(nu_anomes,''), anomes)) ini, max(coalesce(nullif(nu_anomes,''), anomes)) fim
from bronze.bronze_sftp_gehis_andamento_obra
group by 1 order by 2 desc""")

,situacao_obra,n,n_apf,ini,fim
0,PARALISADO,42096,627,,202608
1,NORMAL,40840,1039,,202608
2,OBRA NAO INICIADA,23171,880,202404,202608
3,EM ATENÇÃO,17266,339,,202508
4,EM ATENCAO,13447,542,202410,202608
5,None,4701,1,,
6,LEGALIZACAO,3033,72,,202307
7,,2654,1884,01/04/2023,31/12/2022
8,FAR DEZ 2018,2651,41,,202506
9,RITMO LENTO COVID19,1535,36,,202512


In [8]:
db.ler_sql("select count(*) n, count(distinct nu_apf) n_apf, count(distinct _source_file) n_arquivos, min(_source_file) ini, max(_source_file) fim from bronze.bronze_sftp_gehis_andamento_obra")

,n,n_apf,n_arquivos,ini,fim
0,153545,2225,221,CAIXA_AF_GEHIS_ANDAMENTO_OBRA_M202107.txt,CAIXA_AF_GEHIS_ANDAMENTO_OBRA_M20260821.TXT


## 4. Histórico pré-2024 e paralisações

`prata.prata_hist_far_empreendimento` (331 mil linhas, retratos de várias fontes históricas) traz por empreendimento o físico, o financeiro, `situacao_canonica`, `motivo_paralisacao`, `dt_paralisacao` e datas de contratação, início e conclusão. `ouro.ouro_hist_serie_situacao_mensal` agrega entradas e saídas por situação. É a base antiga de obras paralisadas que o secretário citou.

In [9]:
display(db.ler_sql("select fonte_serie, count(*) n, count(distinct apf) n_apf, min(dt_referencia) ini, max(dt_referencia) fim from prata.prata_hist_far_empreendimento group by 1 order by 2 desc"))
db.ler_sql("select situacao_canonica, count(*) n from prata.prata_hist_far_empreendimento group by 1 order by 2 desc")

,fonte_serie,n,n_apf,ini,fim
0,sftp,235026,4663,2019-12-01,2024-11-01
1,snh,93559,4651,2024-06-01,2026-03-01
2,obra_mensal,2560,876,2026-04-01,2026-07-01


,situacao_canonica,n
0,concluida,272239
1,em_obras,31889
2,cancelada,14769
3,paralisada,5375
4,None,3638
5,nao_iniciada,3235


In [10]:
db.ler_sql("select frente_mcmv, situacao_canonica, min(mes) ini, max(mes) fim, sum(n_empreendimentos) n from ouro.ouro_hist_serie_situacao_mensal where nivel_geografico = 'nacional' group by 1,2 order by 1,5 desc")

,frente_mcmv,situacao_canonica,ini,fim,n
0,Entidades,concluida,2019-12-01,2026-06-01,26063.0
1,Entidades,em_obras,2019-12-01,2026-06-01,9478.0
2,Entidades,paralisada,2019-12-01,2026-06-01,5460.0
3,Entidades,cancelada,2019-12-01,2026-06-01,1766.0
4,Entidades,nao_iniciada,2019-12-01,2026-06-01,816.0
5,FAR,concluida,2019-12-01,2026-03-01,272239.0
6,FAR,em_obras,2019-12-01,2026-03-01,31889.0
7,FAR,cancelada,2019-12-01,2026-03-01,14769.0
8,FAR,paralisada,2019-12-01,2026-03-01,5375.0
9,FAR,nao_iniciada,2019-12-01,2024-11-01,3235.0


## 5. Seeds de domínio úteis

In [11]:
display(db.ler_sql("select * from seeds.far_situacao_obra"))
db.ler_sql("select * from seeds.dominio_motivo_paralisacao")

,co_situacao_obra,situacao_obra
0,1,NÃO INICIADA
1,2,EM ANDAMENTO
2,3,ATRASADA
3,4,PARALISADA
4,5,CONCLUÍDA
5,6,OCUPADA IRREGULARMENTE
6,7,CONCLUÍDA FÍSICA CONCLUÍDA
7,8,CANCELADA OU DISTRATADA
8,9,CONCLUÍDA NÃO ENTREGUE
9,10,CONCLUÍDA E ENTREGUE SEM PENDÊNCIA


,valor_bruto,motivo_paralisacao,origem
0,A ENTREGAR,a_entregar,snh
1,EM ANDAMENTO,em_andamento,snh
2,A RETOMAR,a_retomar,snh
3,A RETOMAR - AO/MCI/ASSINATURA,a_retomar_ao_mci_assinatura,snh
4,A RETOMAR - APORTE ENTE PÚBLICO,a_retomar_aporte_ente_publico,snh
5,A RETOMAR - APORTE SUPLEMENTAR,a_retomar_aporte_suplementar,snh
6,A RETOMAR - EM ANÁLISE,a_retomar_em_analise,snh
7,A RETOMAR - EM CONSTRUÇÃO DA PROPOSTA,a_retomar_em_construcao_proposta,snh
8,A RETOMAR - SOLUÇÃO LOCAL,a_retomar_solucao_local,snh
9,DESIMOBILIZAÇÃO ASSINADA,desimobilizacao_assinada,snh


## 6. O que isso define para o plano

1. **Régua da curva S (FAR novo):** partir de `ouro.ouro_far_serie_mensal_apf` (886 APFs, 2024-02 a 2026-08), que já tem o físico, o financeiro e `dt_inicio_obra` para calcular o mês de obra. A `ouro_far_curva_s` é por mês calendário; a régua por mês de obra é o que vamos construir.
2. **Rótulo da CAIXA:** `bronze.bronze_sftp_gehis_andamento_obra` tipada (APF x anomes x situacao_obra), para validar o rótulo estatístico e estudar as transições NORMAL, EM ATENÇÃO, RISCO DE PARALISAÇÃO, PARALISADO.
3. **Curva S inteira e sobrevivência:** `prata.prata_far_caixa_serie` (4.505 APFs da CAIXA, FAR legado e novo, 2024-06 a 2026-07) e `prata.prata_hist_far_empreendimento` (pré-2024) trazem obras concluídas inteiras e paralisações com motivo.

Próximo notebook (02): base analítica por APF e mês de obra a partir da ouro, com a regra dos 90 dias e o status GEHIS.